# V9ADexternal — one held-out cohort as external validation, with an age validation

**External cohort: GSE132903** (middle temporal gyrus, Illumina HT-12 v4; 97 AD and 98
non-demented donors, one array per donor). It was chosen **before any model was
fitted**, on design grounds read from the merge table alone:

| criterion | GSE132903 |
|---|---|
| one array per person (no within-donor structure) | yes |
| balanced diagnosis | 97 AD / 98 controls |
| age-matched controls | median 86 vs 85 years |
| standard cortical region | middle temporal gyrus |
| independent brain bank and platform | yes |

The alternatives each fail one criterion: GSE33000 has much younger controls (median 68
vs 82) and a 3:1 AD excess; GSE118553 and GSE84422 have 19 and 22 control donors spread
over many regions; GSE48350 is the original discovery cohort.

Models, preprocessing, hyperparameters, attribution, FDR and stability selection are those
of V7ADmerged. The other four cohorts (1,401 arrays) are the training set: 20 real-label
and 10 label-permuted runs, each with a donor-grouped 80/20 train/validation split
(validation only for CVAE early stopping and Integrated Gradients). GSE132903 is touched
once, for the final evaluation: it plays no part in training, early stopping, gene
selection or gene direction.

**Reported on GSE132903:** prediction (ROC-AUC, average precision, balanced accuracy);
replication of the selected genes against random gene sets; the selected genes as a plain
signature score; and an **age validation** — an age-only classifier, the correlation of
each model's predicted risk with age inside each diagnostic group, AUC inside age bands,
whether the models add information beyond age (likelihood-ratio test of AD ~ age vs
AD ~ age + model score), and whether the selected genes are simply age genes.

The label-conditioned CVAE is given each external array's true diagnosis as an input, so
it is reported for reference only; the **label-agnostic CVAE** is the honest result.


In [ ]:
# ======================================================================
# Cell 1 — configuration and data
# ======================================================================
import os, json, warnings, numpy as np, pandas as pd, torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve, confusion_matrix,
                             silhouette_score, adjusted_rand_score,
                             davies_bouldin_score, calinski_harabasz_score)
from scipy import stats
from scipy.stats import rankdata, mannwhitneyu, wilcoxon, ttest_rel

warnings.filterwarnings("ignore")

N_REAL_RUNS = 20          # independent retrained splits, true labels
N_NULL_RUNS = 10          # independent retrained splits, permuted labels
ALPHA       = 0.05        # FDR threshold
IG_STEPS    = 50          # Integrated Gradients path steps
LATENT_DIM  = 8
HIDDEN_DIM  = 256
BETA        = 2.0
GAMMA       = 10.0
MAX_EPOCHS  = 200
PATIENCE    = 20
KL_ANNEAL   = 50
REPRESENTATIVE_SEED = 19  # the split used for single-run illustrations
AGE_MIN     = 60          # V6: age-matched cohort, donors aged >= AGE_MIN only

# Stability-selection windows. The significance test (Mann-Whitney against
# the permuted null) is identical for both methods; only the top-K window is
# widened for the CVAE, whose per-run rankings carry extra estimator variance
# (weight init, batch order, dropout, early-stopping epoch) on top of the
# split variance PLS-DA also sees.
METHOD_PARAMS = {
    "PLS-DA_VIP": {"top_k": 100, "stability_threshold": 0.50},
    "CVAE_IG":    {"top_k": 175, "stability_threshold": 0.40},
}


# ---- V9: single pre-specified external cohort ----------------------------
HOLDOUT = "GSE132903"
LOCO_REAL = 20        # real-label training runs
LOCO_NULL = 10        # label-permuted training runs
N_RANDOM_SETS = 2000  # random gene sets for the replication null
AGE_BANDS = [(60, 79, "<80"), (80, 89, "80-89"), (90, 200, "90+")]
OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

# The published numbers were produced by a session in which CUDA was present but
# its kernels were unusable, so the whole benchmark executed on CPU. PLS-DA is a
# deterministic decomposition and reproduces anywhere; the CVAE does not, because
# dropout masks are drawn from whichever device generator is active. Pinning the
# execution path to CPU is therefore what makes this notebook a reproduction of
# the manuscript rather than a fresh experiment. Set to False to run on GPU --
# expect the CVAE gene list to move, exactly as the paper's Section 4.7 measures.
FORCE_CPU = True


def pick_device():
    """torch.cuda.is_available() is not sufficient on Kaggle: the session can
    be handed a GPU whose compute capability the installed torch build has no
    kernels for, which only surfaces as `no kernel image is available for
    execution on the device` once a real kernel launches. Run a tiny matmul
    and fall back to CPU if it raises."""
    if FORCE_CPU:
        print("FORCE_CPU set; running on CPU as the published run did")
        return torch.device("cpu")
    if torch.cuda.is_available():
        try:
            a = torch.randn(64, 64, device="cuda")
            torch.mm(a, a).sum().item()
            torch.nn.Linear(64, 64).cuda()(a).sum().item()
            return torch.device("cuda")
        except Exception as e:                       # noqa: BLE001
            print(f"CUDA present but unusable ({type(e).__name__}: {e}); using CPU")
    return torch.device("cpu")


device = pick_device()
if device.type == "cpu":
    torch.set_num_threads(os.cpu_count() or 4)
print("device:", device, "| torch", torch.__version__)

# Record the environment rather than leaving it to be inferred from a log. A
# CVAE result is only reproducible against a stated environment, so the versions
# that produced these numbers are part of the output, not part of the metadata.
import platform, sys, importlib


def _ver(mod):
    try:
        return getattr(importlib.import_module(mod), "__version__", "unknown")
    except Exception:                                # noqa: BLE001
        return "not installed"


env = {"python": sys.version.split()[0], "platform": platform.platform(),
       "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__,
       "scikit_learn": _ver("sklearn"), "scipy": _ver("scipy"),
       "device": str(device), "force_cpu": FORCE_CPU,
       "cuda_available": torch.cuda.is_available(),
       "cpu_count": os.cpu_count(), "torch_num_threads": torch.get_num_threads()}
pd.DataFrame([{"key": k, "value": v} for k, v in env.items()]).to_csv(
    f"{OUT}/env_report.csv", index=False)
for k, v in env.items():
    print(f"  {k:16} {v}")

import glob
_hits = glob.glob('/kaggle/input/**/merged_expression.csv.gz', recursive=True)
assert _hits, "attach the ad-merged-geo-cohorts notebook output as an input"
MERGED = os.path.dirname(_hits[0])
print("merged data from", MERGED)

df_genes = pd.read_csv(f"{MERGED}/merged_expression.csv.gz")
df_labels = pd.read_csv(f"{MERGED}/merged_labels.csv").rename(columns={'GSM_ID': 'Subject_ID'})
df_meta = pd.read_csv(f"{MERGED}/merged_metadata.csv")
df = (df_genes.merge(df_labels, on='Subject_ID', how='inner', validate='one_to_one')
              .merge(df_meta, on='Subject_ID', how='inner', validate='one_to_one'))
assert len(df) == len(df_genes) == len(df_labels)
assert (df['GROUP'] == df['dx']).all()
print(pd.read_csv(f"{MERGED}/cohort_table.csv").to_string(index=False))
df = df[df['age'] >= AGE_MIN].reset_index(drop=True)      # already applied upstream
print(f"age >= {AGE_MIN}: {len(df)} arrays from {df.donor.nunique()} donors in "
      f"{df.cohort.nunique()} cohorts")
print(df.groupby(['cohort', 'GROUP']).agg(arrays=('Subject_ID', 'size'),
                                          donors=('donor', 'nunique')).to_string())

meta_cols   = ['Subject_ID', 'GROUP', 'donor', 'region', 'age', 'sex', 'dx', 'cohort', 'platform']
gene_cols   = [c for c in df.columns if c not in meta_cols]
donor_ids   = df['donor'].values
regions     = df['region'].values
ages        = df['age'].values.astype(float)
cohorts     = df['cohort'].values
subject_ids = df['Subject_ID'].values
X_raw       = df[gene_cols].values.astype(np.float32)
label_encoder = LabelEncoder()
y_raw       = label_encoder.fit_transform(df['GROUP'].values)
num_classes = len(label_encoder.classes_)
n_genes     = len(gene_cols)
AD_IDX      = int(np.where(label_encoder.classes_ == 'AD')[0][0])

print(f"arrays={X_raw.shape[0]}  donors={len(np.unique(donor_ids))}  genes={n_genes}  "
      f"classes={list(label_encoder.classes_)}")
print(pd.Series(df['GROUP']).value_counts().to_string())

pd.DataFrame({
    "n_subjects": [X_raw.shape[0]], "n_genes": [n_genes],
    "n_AD": [int((y_raw == AD_IDX).sum())],
    "n_control": [int((y_raw != AD_IDX).sum())],
    "class_order": ["|".join(label_encoder.classes_)],
    "AD_index": [AD_IDX],
    "n_donors": [len(np.unique(donor_ids))],
    "n_donors_AD": [len(np.unique(donor_ids[y_raw == AD_IDX]))],
    "n_donors_control": [len(np.unique(donor_ids[y_raw != AD_IDX]))],
    "age_min": [AGE_MIN],
    "n_cohorts": [len(np.unique(cohorts))],
    "age_median_AD": [float(np.median(df.loc[y_raw == AD_IDX].drop_duplicates('donor').age))],
    "age_median_control": [float(np.median(df.loc[y_raw != AD_IDX].drop_duplicates('donor').age))],
}).to_csv(f"{OUT}/cohort_summary.csv", index=False)
df[['Subject_ID', 'GROUP', 'cohort', 'platform', 'donor', 'region', 'age', 'sex']].to_csv(
    f"{OUT}/sample_metadata.csv", index=False)

In [ ]:
# ======================================================================
# Cell 2 — model, losses and shared helpers
# ======================================================================
class GeneExpressionDataset(Dataset):
    def __init__(self, X, y, num_classes):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y).long()
        self.num_classes = num_classes
    def __len__(self):  return self.X.shape[0]
    def __getitem__(self, i):
        y = self.y[i]
        cond = torch.nn.functional.one_hot(y, num_classes=self.num_classes).float()
        return self.X[i], cond, y


class CVAE(nn.Module):
    def __init__(self, input_dim, cond_dim, latent_dim=8, hidden_dim=256, num_classes=2):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim + cond_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(0.2))
        self.mu_layer     = nn.Linear(hidden_dim, latent_dim)
        self.logvar_layer = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim + cond_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, input_dim))
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_classes))

    def encode(self, x, cond):
        h = self.encoder(torch.cat([x, cond], dim=1))
        return self.mu_layer(h), self.logvar_layer(h)

    def reparameterize(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)

    def decode(self, z, cond):
        return self.decoder(torch.cat([z, cond], dim=1))

    def forward(self, x, cond):
        mu, logvar = self.encode(x, cond)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, cond), mu, logvar, self.classifier(mu)


def cvae_loss(recon_x, x, mu, logvar, logits, y, beta, gamma):
    recon = nn.MSELoss(reduction='mean')(recon_x, x)
    kl    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    ce    = nn.CrossEntropyLoss()(logits, y)
    return recon + beta * kl + gamma * ce


def neutral_cond(n, num_classes, device):
    """Label-agnostic conditioning: uniform over classes, so the encoder is
    given no information about the subject's diagnosis."""
    return torch.full((n, num_classes), 1.0 / num_classes, device=device)


def encode_all(model, X, y, num_classes, device, mode="true", batch_size=256):
    """Return (mu, logvar) for every row of X. mode='true' conditions on the
    observed label; mode='neutral' conditions on the uniform vector."""
    model.eval()
    mus, lvs = [], []
    with torch.no_grad():
        for s in range(0, len(X), batch_size):
            xb = torch.tensor(X[s:s + batch_size], dtype=torch.float32, device=device)
            if mode == "true":
                yb = torch.tensor(y[s:s + batch_size], dtype=torch.long, device=device)
                cb = torch.nn.functional.one_hot(yb, num_classes).float()
            else:
                cb = neutral_cond(len(xb), num_classes, device)
            mu, lv = model.encode(xb, cb)
            mus.append(mu.cpu().numpy()); lvs.append(lv.cpu().numpy())
    return np.vstack(mus), np.vstack(lvs)


def reconstruct_cvae(model, X, y, num_classes, device, mode="true", batch_size=256):
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(X), batch_size):
            xb = torch.tensor(X[s:s + batch_size], dtype=torch.float32, device=device)
            if mode == "true":
                yb = torch.tensor(y[s:s + batch_size], dtype=torch.long, device=device)
                cb = torch.nn.functional.one_hot(yb, num_classes).float()
            else:
                cb = neutral_cond(len(xb), num_classes, device)
            mu, _ = model.encode(xb, cb)
            out.append(model.decode(mu, cb).cpu().numpy())   # decode the mean
    return np.vstack(out)


def per_sample_kl(mu, logvar):
    """KL(q(z|x,c) || N(0,I)) per subject, summed over latent dimensions (nats)."""
    return -0.5 * np.sum(1.0 + logvar - mu ** 2 - np.exp(logvar), axis=1)


def cluster_metrics(Z, y):
    km = KMeans(n_clusters=len(np.unique(y)), n_init=10, random_state=42)
    labels = km.fit_predict(Z)
    return {
        "silhouette":        silhouette_score(Z, y),
        "ari":               adjusted_rand_score(y, labels),
        # V6: scored against the diagnosis, like silhouette (v5 used the
        # k-means partition, which measures k-means rather than diagnosis)
        "davies_bouldin":    davies_bouldin_score(Z, y),
        "calinski_harabasz": calinski_harabasz_score(Z, y),
    }


def inter_intra_ratio(Z, y):
    """Mean between-class centroid distance over mean within-class spread."""
    cents = np.array([Z[y == c].mean(axis=0) for c in np.unique(y)])
    inter = np.linalg.norm(cents[0] - cents[1])
    intra = np.mean([np.linalg.norm(Z[y == c] - cents[i], axis=1).mean()
                     for i, c in enumerate(np.unique(y))])
    return inter / (intra + 1e-12)


def vip_scores(pls):
    """Variable Importance in Projection (Wold et al. 2001): a gene's weight
    aggregated over all components, weighted by the Y-variance each explains."""
    t, w, q = pls.x_scores_, pls.x_weights_, pls.y_loadings_
    p, _ = w.shape
    s = np.sum(t ** 2, axis=0) * np.sum(q ** 2, axis=0)
    w_norm = w / (np.linalg.norm(w, axis=0, keepdims=True) + 1e-12)
    return np.sqrt(p * (w_norm ** 2) @ s / (np.sum(s) + 1e-12))


def bh_fdr(pvals):
    pvals = np.asarray(pvals); n = len(pvals)
    order = np.argsort(pvals)
    q = pvals[order] * n / np.arange(1, n + 1)
    q = np.clip(np.minimum.accumulate(q[::-1])[::-1], 0, 1)
    out = np.empty(n); out[order] = q
    return out

print("helpers defined")

In [ ]:
# ======================================================================
# Cell 3 — Integrated Gradients
# ======================================================================
!pip install captum umap-learn -q
from captum.attr import IntegratedGradients


def ig_attributions(model, X, y, num_classes, device, n_steps=IG_STEPS):
    """Attribute the true-class classifier logit back to genes, relative to a
    zero (= training-mean, post-standardisation) baseline."""
    model.eval()
    X_t = torch.tensor(X, dtype=torch.float32, device=device)
    y_t = torch.tensor(y, dtype=torch.long, device=device)
    cond = torch.nn.functional.one_hot(y_t, num_classes).float()

    def forward_fn(x, cond_arg):
        mu, _ = model.encode(x, cond_arg)
        return model.classifier(mu)

    # cond must travel through additional_forward_args: captum expands
    # `inputs` to (batch * n_steps, ...) for the path integral and only
    # applies that expansion to tensors it is told about.
    # internal_batch_size caps that expansion — without it the path integral
    # materialises n_test * n_steps * n_genes floats in one allocation.
    attr = IntegratedGradients(forward_fn).attribute(
        X_t, torch.zeros_like(X_t), target=y_t,
        additional_forward_args=(cond,), n_steps=n_steps,
        internal_batch_size=64)
    return attr.detach().cpu().numpy()


def cvae_latent_saliency(model, X, y, num_classes, device, mode="true", batch_size=128):
    """|d mu_k / d x_j| averaged over subjects — the latent-to-gene map."""
    model.eval()
    accum = np.zeros((model.latent_dim, X.shape[1]), dtype=np.float64)
    n_seen = 0
    for s in range(0, len(X), batch_size):
        xb = torch.tensor(X[s:s + batch_size], dtype=torch.float32,
                          device=device, requires_grad=True)
        if mode == "true":
            yb = torch.tensor(y[s:s + batch_size], dtype=torch.long, device=device)
            cb = torch.nn.functional.one_hot(yb, num_classes).float()
        else:
            cb = neutral_cond(len(xb), num_classes, device)
        mu, _ = model.encode(xb, cb)
        for k in range(model.latent_dim):
            g = torch.autograd.grad(mu[:, k].sum(), xb, retain_graph=(k < model.latent_dim - 1))[0]
            accum[k] += g.abs().sum(dim=0).detach().cpu().numpy()
        n_seen += len(xb)
    return accum / max(n_seen, 1)

print("attribution helpers ready")

In [ ]:
# ======================================================================
# V8 Cell 4 — training-pool split and model fitting
# ======================================================================
def pool_split(pool_idx, seed, permute=False):
    """Donor-grouped 80/20 train/validation split of the training cohorts,
    stratified by cohort x diagnosis. permute=True permutes diagnoses across
    donors within each cohort (the label-permuted null)."""
    d_ids = donor_ids[pool_idx]
    lab = pd.Series(y_raw[pool_idx]).groupby(d_ids).first()
    coh = pd.Series(cohorts[pool_idx]).groupby(d_ids).first()
    if permute:
        rs = np.random.RandomState(seed + 999_000)
        perm = lab.copy()
        for c in sorted(coh.unique()):
            ids = coh.index[coh == c]
            perm.loc[ids] = rs.permutation(lab.loc[ids].values)
        lab = perm
    strata = coh.astype(str) + "|" + lab.astype(str)
    d_tr, d_va = train_test_split(lab.index.values, test_size=0.2,
                                  stratify=strata.loc[lab.index].values, random_state=seed)
    idx_tr = pool_idx[np.isin(d_ids, d_tr)]
    idx_va = pool_idx[np.isin(d_ids, d_va)]
    assert not (set(donor_ids[idx_tr]) & set(donor_ids[idx_va])), "donor leak train/val"
    y_arr = pd.Series(lab.loc[donor_ids[pool_idx]].values, index=pool_idx)
    return idx_tr, idx_va, y_arr.loc[idx_tr].values, y_arr.loc[idx_va].values


def fit_models(X_tr, y_tr, X_va, y_va, seed):
    """PLS-DA and the beta-CVAE with exactly the V5-V7 settings."""
    np.random.seed(seed); torch.manual_seed(seed)
    ohe = OneHotEncoder(sparse_output=False)
    plsda = PLSRegression(n_components=LATENT_DIM)
    plsda.fit(X_tr, ohe.fit_transform(y_tr.reshape(-1, 1)))

    tr_loader = DataLoader(GeneExpressionDataset(X_tr, y_tr, num_classes), batch_size=32,
                           shuffle=True, drop_last=(len(X_tr) % 32 == 1))
    va_loader = DataLoader(GeneExpressionDataset(X_va, y_va, num_classes), batch_size=32,
                           shuffle=False)
    model = CVAE(X_tr.shape[1], num_classes, LATENT_DIM, HIDDEN_DIM, num_classes).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=10)
    best_acc, best_state, waited, stopped_at = 0.0, None, 0, MAX_EPOCHS
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        kl_w = min(1.0, epoch / KL_ANNEAL)
        for x, cond, yb in tr_loader:
            x, cond, yb = x.to(device), cond.to(device), yb.to(device)
            opt.zero_grad()
            recon, mu, logvar, logits = model(x, cond)
            cvae_loss(recon, x, mu, logvar, logits, yb, BETA * kl_w, GAMMA).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
        model.eval(); preds, true = [], []
        with torch.no_grad():
            for x, cond, yb in va_loader:
                _, _, _, logits = model(x.to(device), cond.to(device))
                preds.extend(torch.argmax(logits, 1).cpu().numpy()); true.extend(yb.numpy())
        val_acc = accuracy_score(true, preds)
        sched.step(val_acc)
        if val_acc > best_acc + 1e-3:
            best_acc, best_state, waited = val_acc, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            waited += 1
            if waited >= PATIENCE:
                stopped_at = epoch; break
    if best_state is not None:
        model.load_state_dict(best_state)
    return plsda, model, best_acc, stopped_at

print("LOCO trainer ready")


In [ ]:
# ======================================================================
# V9 Cell 5 — train on four cohorts, evaluate on the external one
# ======================================================================
import time
COHORTS = [HOLDOUT]
BASE_FPR = np.linspace(0, 1, 101)
loco_runs, loco_roc, loco_pred = [], [], []
ATTR = {c: {"vip_real": [], "ig_real": [], "vip_null": [], "ig_null": []} for c in COHORTS}


def lr_prob(Z_tr, y_tr, Z_te):
    clf = LogisticRegression(max_iter=1000).fit(Z_tr, y_tr)
    return clf.predict_proba(Z_te)[:, list(clf.classes_).index(AD_IDX)]


def donor_auc(y_ad, prob, d_ids):
    dy = pd.Series(y_ad).groupby(d_ids).first()
    dp = pd.Series(prob).groupby(d_ids).mean().loc[dy.index]
    return roc_auc_score(dy.values, dp.values) if dy.nunique() == 2 else np.nan


t0 = time.time()
for c in COHORTS:
    ext = np.where(cohorts == c)[0]
    pool = np.where(cohorts != c)[0]
    y_ext = y_raw[ext]; y_ad = (y_ext == AD_IDX).astype(int)
    print(f"\n=== held out: {c}  ({len(ext)} arrays, {len(np.unique(donor_ids[ext]))} donors; "
          f"training on {len(pool)} arrays from {len(np.unique(cohorts[pool]))} cohorts)")
    for r in range(LOCO_REAL + LOCO_NULL):
        null = r >= LOCO_REAL
        seed = (10_000 + r) if null else r
        idx_tr, idx_va, y_tr, y_va = pool_split(pool, seed, permute=null)
        scaler = StandardScaler().fit(X_raw[idx_tr])
        X_tr = scaler.transform(X_raw[idx_tr]).astype(np.float32)
        X_va = scaler.transform(X_raw[idx_va]).astype(np.float32)
        plsda, model, best_acc, ep = fit_models(X_tr, y_tr, X_va, y_va, seed)
        key = "null" if null else "real"
        ATTR[c][f"vip_{key}"].append(vip_scores(plsda))
        ATTR[c][f"ig_{key}"].append(np.abs(ig_attributions(model, X_va, y_va, num_classes,
                                                            device)).mean(axis=0))
        if null:
            print(f"  null {r - LOCO_REAL + 1}/{LOCO_NULL}"); continue

        # ---- external evaluation: the held-out cohort, never seen ------
        X_ex = scaler.transform(X_raw[ext]).astype(np.float32)
        prob = {"pls": lr_prob(plsda.transform(X_tr), y_tr, plsda.transform(X_ex))}
        for mode in ("neutral", "true"):
            mu_tr, _ = encode_all(model, X_tr, y_tr, num_classes, device, mode)
            mu_ex, _ = encode_all(model, X_ex, y_ext, num_classes, device, mode)
            prob[f"cvae_{mode}"] = lr_prob(mu_tr, y_tr, mu_ex)
        prob["age"] = lr_prob(ages[idx_tr].reshape(-1, 1), y_tr, ages[ext].reshape(-1, 1))

        row = {"held_out": c, "run": r + 1, "seed": seed, "n_ext": len(ext),
               "n_train": len(idx_tr), "n_val": len(idx_va), "epochs_run": ep,
               "best_val_acc": best_acc}
        for m, p in prob.items():
            row[f"auc_{m}"] = roc_auc_score(y_ad, p)
            row[f"ap_{m}"] = average_precision_score(y_ad, p)
            pred = (p >= 0.5).astype(int)
            sens = pred[y_ad == 1].mean(); spec = 1 - pred[y_ad == 0].mean()
            row[f"bacc_{m}"] = 0.5 * (sens + spec)
            row[f"auc_{m}_donor"] = donor_auc(y_ad, p, donor_ids[ext])
            if True:   # V9: keep the age-only ROC too
                fpr, tpr, _ = roc_curve(y_ad, p)
                loco_roc.append({"held_out": c, "run": r + 1, "model": m,
                                 **{f"tpr_{i}": v for i, v in enumerate(np.interp(BASE_FPR, fpr, tpr))}})
        loco_runs.append(row)
        for i, g in enumerate(ext):
            loco_pred.append({"held_out": c, "run": r + 1, "subject_id": subject_ids[g],
                              "donor": donor_ids[g], "region": regions[g],
                              "group": label_encoder.classes_[y_ext[i]],
                              **{f"p_{m}": float(p[i]) for m, p in prob.items()}})
        print(f"  run {r+1:2d}/{LOCO_REAL}  AUC  PLS-DA {row['auc_pls']:.3f} | CVAE agnostic "
              f"{row['auc_cvae_neutral']:.3f} | CVAE conditioned* {row['auc_cvae_true']:.3f} | "
              f"age {row['auc_age']:.3f}   [{(time.time()-t0)/60:.1f} min]")

runs = pd.DataFrame(loco_runs)
runs.to_csv(f"{OUT}/loco_run_metrics.csv", index=False)
pd.DataFrame(loco_roc).to_csv(f"{OUT}/loco_roc_curves.csv", index=False)
pd.DataFrame(loco_pred).to_csv(f"{OUT}/loco_predictions.csv", index=False)
for c in COHORTS:
    for k, v in ATTR[c].items():
        np.save(f"{OUT}/loco_{k}_{c}.npy", np.array(v))
print("\n* the conditioned arm is given each held-out array's true diagnosis -- reference only")


In [ ]:
# ======================================================================
# Cell 7 — gene significance, stability, direction
# ======================================================================
def summarize(method, real, null, genes, top_k, stability_threshold):
    """Rank-based test against the permuted null.

    Magnitudes cannot be compared directly: mean(VIP^2) equals 1 for any
    fitted PLS model by construction, and a network trained on permuted
    labels can still produce large IG attributions by memorising noise.
    Ranking within each run sidesteps both — a gene consistently top-ranked
    across real runs but randomly ranked across null runs is real signal.
    """
    n_real, n_g = real.shape
    real_ranks = np.array([rankdata(-real[i]) for i in range(n_real)])
    null_ranks = np.array([rankdata(-null[i]) for i in range(null.shape[0])])

    pvals = np.ones(n_g)
    for j in range(n_g):
        try:
            pvals[j] = mannwhitneyu(real_ranks[:, j], null_ranks[:, j], alternative='less')[1]
        except ValueError:
            pvals[j] = 1.0
    qvals = bh_fdr(pvals)

    flags = np.zeros_like(real, dtype=bool)
    for i in range(n_real):
        flags[i, np.argsort(real[i])[::-1][:top_k]] = True
    stability = flags.mean(axis=0)

    out = pd.DataFrame({
        "gene": genes, "mean_score": real.mean(axis=0), "std_score": real.std(axis=0),
        "p_value": pvals, "fdr_q": qvals, "stability_freq": stability,
        "significant": (qvals < ALPHA) & (stability >= stability_threshold),
    }).sort_values("mean_score", ascending=False).reset_index(drop=True)
    out.insert(0, "method", method)
    return out




In [ ]:
# ======================================================================
# V9 Cell 6 — gene selection on the training cohorts, replication on the held-out one
# ======================================================================
def de_stats(X, y):
    ad = (y == AD_IDX)
    lfc = X[ad].mean(0) - X[~ad].mean(0)
    p = np.array([mannwhitneyu(X[ad, j], X[~ad, j], alternative="two-sided")[1]
                  for j in range(X.shape[1])])
    return lfc, p


rng = np.random.RandomState(0)
gene_rows, rep_rows, sig_rows = [], [], []
for c in COHORTS:
    ext = np.where(cohorts == c)[0]; pool = np.where(cohorts != c)[0]
    A = ATTR[c]
    stats_ = {"PLS-DA": summarize("PLS-DA_VIP", np.array(A["vip_real"]), np.array(A["vip_null"]),
                                  gene_cols, **METHOD_PARAMS["PLS-DA_VIP"]),
              "CVAE": summarize("CVAE_IG", np.array(A["ig_real"]), np.array(A["ig_null"]),
                                gene_cols, **METHOD_PARAMS["CVAE_IG"])}
    # direction is learned on the training cohorts only, with the project's rule
    lfc_tr, p_tr = de_stats(X_raw[pool], y_raw[pool])
    dir_tr = np.where(p_tr < 0.10, np.sign(lfc_tr), 0)
    # the held-out cohort's own differential expression
    lfc_ex, p_ex = de_stats(X_raw[ext], y_raw[ext])
    q_ex = bh_fdr(p_ex)
    gi = {g: i for i, g in enumerate(gene_cols)}
    directional = np.where(dir_tr != 0)[0]

    def replication(idx):
        if len(idx) == 0:
            return np.nan, np.nan
        conc = np.sign(lfc_ex[idx]) == dir_tr[idx]
        return conc.mean(), (conc & (q_ex[idx] < 0.05)).mean()

    for method, st in stats_.items():
        sel = st[st.significant].gene.tolist()
        idx = np.array([gi[g] for g in sel if dir_tr[gi[g]] != 0], dtype=int)
        for g in sel:
            j = gi[g]
            gene_rows.append({"held_out": c, "method": method, "gene": g,
                              "direction_train": {1: "up", -1: "down", 0: "none"}[int(dir_tr[j])],
                              "stability_freq": float(st.set_index("gene").stability_freq[g]),
                              "fdr_q_train": float(st.set_index("gene").fdr_q[g]),
                              "logFC_heldout": lfc_ex[j], "q_heldout": q_ex[j],
                              "concordant": bool(np.sign(lfc_ex[j]) == dir_tr[j]) if dir_tr[j] else None,
                              "replicated": bool(np.sign(lfc_ex[j]) == dir_tr[j] and q_ex[j] < 0.05) if dir_tr[j] else None})
        conc, repl = replication(idx)
        # null: random directional genes, same number, same rule
        null_conc, null_repl = [], []
        for _ in range(N_RANDOM_SETS):
            if len(idx) == 0:
                break
            rnd = rng.choice(directional, size=len(idx), replace=False)
            a, b = replication(rnd); null_conc.append(a); null_repl.append(b)
        null_conc, null_repl = np.array(null_conc), np.array(null_repl)
        rep_rows.append({"held_out": c, "method": method, "n_selected": len(sel),
                         "n_directional": len(idx), "concordance": conc, "replication": repl,
                         "null_concordance_mean": null_conc.mean() if len(null_conc) else np.nan,
                         "null_concordance_lo": np.percentile(null_conc, 2.5) if len(null_conc) else np.nan,
                         "null_concordance_hi": np.percentile(null_conc, 97.5) if len(null_conc) else np.nan,
                         "null_replication_mean": null_repl.mean() if len(null_repl) else np.nan,
                         "null_replication_lo": np.percentile(null_repl, 2.5) if len(null_repl) else np.nan,
                         "null_replication_hi": np.percentile(null_repl, 97.5) if len(null_repl) else np.nan,
                         "p_concordance": (1 + (null_conc >= conc).sum()) / (1 + len(null_conc)) if len(null_conc) else np.nan,
                         "p_replication": (1 + (null_repl >= repl).sum()) / (1 + len(null_repl)) if len(null_repl) else np.nan})
        # signature score: mean(up) - mean(down) on the held-out arrays, no model
        up = [gi[g] for g in sel if dir_tr[gi[g]] > 0]
        dn = [gi[g] for g in sel if dir_tr[gi[g]] < 0]
        Xe = X_raw[ext]
        score = (Xe[:, up].mean(1) if up else 0) - (Xe[:, dn].mean(1) if dn else 0)
        y_ad = (y_raw[ext] == AD_IDX).astype(int)
        ok = bool(up or dn)
        sig_rows.append({"held_out": c, "method": method, "n_up": len(up), "n_down": len(dn),
                         "signature_auc": roc_auc_score(y_ad, score) if ok else np.nan,
                         "signature_auc_donor": donor_auc(y_ad, score, donor_ids[ext]) if ok else np.nan})
    print(f"{c}: PLS-DA {len(stats_['PLS-DA'][stats_['PLS-DA'].significant])} genes, "
          f"CVAE {len(stats_['CVAE'][stats_['CVAE'].significant])} genes selected on the other four")

genes_df = pd.DataFrame(gene_rows, columns=["held_out", "method", "gene", "direction_train", "stability_freq", "fdr_q_train", "logFC_heldout", "q_heldout", "concordant", "replicated"])
genes_df.to_csv(f"{OUT}/loco_selected_genes.csv", index=False)
rep_df = pd.DataFrame(rep_rows); rep_df.to_csv(f"{OUT}/loco_gene_replication.csv", index=False)
sig_df = pd.DataFrame(sig_rows); sig_df.to_csv(f"{OUT}/loco_signature_scores.csv", index=False)
print(rep_df.round(3).to_string(index=False))
print(sig_df.round(3).to_string(index=False))

# genes selected in several folds: the cohort-robust part of each list
freq = (genes_df.groupby(["method", "gene"]).held_out.nunique().rename("n_folds").reset_index())
rep_rate = genes_df.dropna(subset=["replicated"]).groupby(["method", "gene"]).replicated.mean()
freq = freq.merge(rep_rate.rename("replicated_frac").reset_index(), how="left")
freq.sort_values(["method", "n_folds"], ascending=[True, False]).to_csv(
    f"{OUT}/loco_gene_frequency.csv", index=False)
print(freq.groupby("method").n_folds.value_counts().sort_index().to_string())


In [ ]:
# ======================================================================
# V9 Cell 8 — age validation on the external cohort
# ======================================================================
from scipy.stats import spearmanr, chi2, fisher_exact
ext = np.where(cohorts == HOLDOUT)[0]
pool = np.where(cohorts != HOLDOUT)[0]
pred = pd.DataFrame(loco_pred)
mean_p = pred.groupby("subject_id")[["p_pls", "p_cvae_neutral", "p_cvae_true", "p_age"]].mean()
mean_p = mean_p.loc[subject_ids[ext]]
y_ad = (y_raw[ext] == AD_IDX).astype(int)
age_e = ages[ext]
ARMS = [("pls", "PLS-DA"), ("cvae_neutral", "CVAE (label-agnostic)"), ("cvae_true", "CVAE (label-conditioned)*")]

age_rows = []
put_age = lambda k, v: age_rows.append({"key": k, "value": v})
put_age("n_external", len(ext)); put_age("n_AD", int(y_ad.sum())); put_age("n_control", int((1 - y_ad).sum()))
put_age("age_median_AD", float(np.median(age_e[y_ad == 1])))
put_age("age_median_control", float(np.median(age_e[y_ad == 0])))
put_age("age_mannwhitney_p", mannwhitneyu(age_e[y_ad == 1], age_e[y_ad == 0])[1])
put_age("auc_age_only_raw", roc_auc_score(y_ad, age_e))        # age itself, no model


def loglik(X, y):
    """Unpenalised logistic regression log-likelihood."""
    clf = LogisticRegression(penalty=None, max_iter=5000).fit(X, y)
    p = np.clip(clf.predict_proba(X)[:, 1], 1e-12, 1 - 1e-12)
    return np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)), clf


logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
ll_age, _ = loglik(age_e.reshape(-1, 1), y_ad)
for m, lab in ARMS:
    p = mean_p[f"p_{m}"].values
    for g, gname in ((0, "control"), (1, "AD")):
        rho, pr = spearmanr(p[y_ad == g], age_e[y_ad == g])
        put_age(f"rho_age_{m}_{gname}", rho); put_age(f"rho_age_{m}_{gname}_p", pr)
    # does the model add information beyond age?  AD ~ age  vs  AD ~ age + score
    ll_both, clf = loglik(np.column_stack([age_e, logit(p)]), y_ad)
    lr = 2 * (ll_both - ll_age)
    put_age(f"lr_{m}_beyond_age_stat", lr); put_age(f"lr_{m}_beyond_age_p", chi2.sf(lr, 1))
    put_age(f"auc_{m}_plus_age_insample", roc_auc_score(y_ad, clf.predict_proba(
        np.column_stack([age_e, logit(p)]))[:, 1]))
    put_age(f"auc_{m}_mean_prob", roc_auc_score(y_ad, p))

band_rows = []
for lo, hi, name in AGE_BANDS:
    sel = (age_e >= lo) & (age_e <= hi)
    r = {"band": name, "n": int(sel.sum()), "n_AD": int(y_ad[sel].sum()),
         "n_control": int((1 - y_ad[sel]).sum()),
         "age_median_AD": float(np.median(age_e[sel & (y_ad == 1)])) if (sel & (y_ad == 1)).any() else np.nan,
         "age_median_control": float(np.median(age_e[sel & (y_ad == 0)])) if (sel & (y_ad == 0)).any() else np.nan}
    for m, _ in ARMS + [("age", "age")]:
        aucs = []
        for _, rr in pred.groupby("run"):
            rr = rr.set_index("subject_id").loc[subject_ids[ext]]
            yy, pp = y_ad[sel], rr[f"p_{m}"].values[sel]
            if yy.min() != yy.max():
                aucs.append(roc_auc_score(yy, pp))
        r[f"auc_{m}_mean"] = np.mean(aucs) if aucs else np.nan
        r[f"auc_{m}_sd"] = np.std(aucs, ddof=1) if len(aucs) > 1 else np.nan
    band_rows.append(r)
bands = pd.DataFrame(band_rows)
bands.to_csv(f"{OUT}/external_age_bands.csv", index=False)

# are the selected genes just ageing genes?  age association in TRAINING controls only
ctrl_pool = pool[y_raw[pool] != AD_IDX]
Xc, ac = X_raw[ctrl_pool], ages[ctrl_pool]
_sp = [spearmanr(Xc[:, j], ac) for j in range(n_genes)]
rho_g = np.array([r[0] for r in _sp]); p_g = np.array([r[1] for r in _sp])
q_g = bh_fdr(np.nan_to_num(p_g, nan=1.0))
gene_age = pd.DataFrame({"gene": gene_cols, "rho_age_train_controls": rho_g, "p": p_g, "q": q_g})
gene_age.to_csv(f"{OUT}/gene_age_association.csv", index=False)
bg = (q_g < 0.05).mean()
put_age("frac_age_genes_background", bg)
sel_g = pd.read_csv(f"{OUT}/loco_selected_genes.csv")
for method in ("PLS-DA", "CVAE"):
    gs_ = set(sel_g[sel_g.method == method].gene)
    mask = gene_age.gene.isin(gs_).values
    k, n = int(((q_g < 0.05) & mask).sum()), int(mask.sum())
    put_age(f"frac_age_genes_{method}", k / n if n else np.nan)
    put_age(f"n_age_genes_{method}", k); put_age(f"n_selected_{method}", n)
    if n:
        tab = [[k, n - k], [int((q_g < 0.05).sum()) - k, int((q_g >= 0.05).sum()) - (n - k)]]
        put_age(f"age_genes_{method}_fisher_p", fisher_exact(tab)[1])

age_df = pd.DataFrame(age_rows)
age_df.to_csv(f"{OUT}/external_age_validation.csv", index=False)
mean_p.assign(group=label_encoder.classes_[y_raw[ext]], age=age_e).to_csv(
    f"{OUT}/external_mean_predictions.csv")
print(age_df.to_string(index=False))
print(bands.round(3).to_string(index=False))


In [ ]:
# ======================================================================
# V9 Cell 9 — the report, in one table
# ======================================================================
S9 = dict(zip(age_df.key, age_df.value))
runs9 = pd.DataFrame(loco_runs)
rep9 = pd.read_csv(f"{OUT}/loco_gene_replication.csv")
sig9 = pd.read_csv(f"{OUT}/loco_signature_scores.csv")
lines = [f"EXTERNAL VALIDATION on {HOLDOUT}: {S9['n_external']:.0f} arrays "
         f"({S9['n_AD']:.0f} AD / {S9['n_control']:.0f} control), trained on the other four cohorts", ""]
for m, lab in ARMS + [("age", "age only (trained)")]:
    lines.append(f"  {lab:30s} AUC {runs9[f'auc_{m}'].mean():.3f} +/- {runs9[f'auc_{m}'].std():.3f}   "
                 f"AP {runs9[f'ap_{m}'].mean():.3f}   balanced acc {runs9[f'bacc_{m}'].mean():.3f}")
t, p = ttest_rel(runs9.auc_cvae_neutral, runs9.auc_pls)
lines += [f"  CVAE (label-agnostic) vs PLS-DA, paired over 20 runs: p = {p:.2g}", "",
          "AGE VALIDATION",
          f"  median age AD {S9['age_median_AD']:.0f} vs control {S9['age_median_control']:.0f} "
          f"(Mann-Whitney p = {S9['age_mannwhitney_p']:.2g}); raw age AUC {S9['auc_age_only_raw']:.3f}"]
for m, lab in ARMS[:2]:
    lines.append(f"  {lab:24s} risk vs age: rho controls {S9[f'rho_age_{m}_control']:+.2f} "
                 f"(p {S9[f'rho_age_{m}_control_p']:.2g}), rho AD {S9[f'rho_age_{m}_AD']:+.2f} "
                 f"(p {S9[f'rho_age_{m}_AD_p']:.2g}); beyond age: LR p = {S9[f'lr_{m}_beyond_age_p']:.2g}")
for _, b in bands.iterrows():
    lines.append(f"  age {b.band:6s} n={b.n:3d} ({b.n_AD} AD/{b.n_control} C): AUC PLS-DA "
                 f"{b.auc_pls_mean:.3f}, CVAE {b.auc_cvae_neutral_mean:.3f}, age {b.auc_age_mean:.3f}")
lines.append(f"  age-associated genes (training controls, FDR<0.05): background "
             f"{S9['frac_age_genes_background']:.1%}, PLS-DA set {S9.get('frac_age_genes_PLS-DA', np.nan):.1%}, "
             f"CVAE set {S9.get('frac_age_genes_CVAE', np.nan):.1%}")
lines += ["", "GENES SELECTED ON THE FOUR TRAINING COHORTS, TESTED ON THE EXTERNAL ONE"]
for _, r in rep9.iterrows():
    s_ = sig9[sig9.method == r.method].iloc[0]
    lines.append(f"  {r.method:7s} {r.n_selected:3.0f} genes: same direction {r.concordance:.1%} "
                 f"(random {r.null_concordance_mean:.1%}, p {r.p_concordance:.3g}); "
                 f"replicated at FDR<0.05 {r.replication:.1%} (random {r.null_replication_mean:.1%}, "
                 f"p {r.p_replication:.3g}); signature AUC {s_.signature_auc:.3f}")
report = "\n".join(lines)
open(f"{OUT}/EXTERNAL_VALIDATION_REPORT.txt", "w").write(report + "\n")
print(report)


---

# Figures

Same visual language as V5-V8 (the project's `figstyle` module, verbatim). Every figure is drawn from the CSVs written above and shown inline.


In [ ]:
# Point the shared style module at this notebook's output directory and
# ask it to render inline, then run it exactly as the paper build does.
import os
os.environ["PAPER_DATA"] = OUT
os.environ["PAPER_FIGS"] = f"{OUT}/figures"
os.environ["PAPER_INLINE"] = "1"

"""Shared visual language for every figure in the paper.

One module so that a panel letter, a tick, a bar gap or a significance bracket
looks the same wherever it appears -- the thing that separates a figure set from
a pile of plots. Everything here is layout and typography; the numbers come from
../data and are never touched.

Colour is the data-viz reference palette used unchanged: categorical slots 1-3
(blue / orange / aqua) for the three model arms, which is the documented
all-pairs colourblind-safe subset, and the documented blue-red diverging pair
with a neutral grey midpoint for up/down regulation. Identity never rests on
colour alone -- every series also carries a marker shape, a direct label or a
legend entry.
"""
import os
import warnings
from pathlib import Path

import matplotlib as mpl
# PAPER_INLINE is set when this same code runs inside the Kaggle notebook, where
# the figures must render in the output rather than only be written to disk.
INLINE = bool(os.environ.get("PAPER_INLINE"))
if not INLINE:
    mpl.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")

# Paths are overridable so that the notebook can point them at /kaggle/working
# and run this file unmodified -- one source of truth for every figure.
if "PAPER_DATA" in os.environ:
    DATA = Path(os.environ["PAPER_DATA"])
    FIG = Path(os.environ.get("PAPER_FIGS", DATA / "figures"))
else:
    ROOT = Path(__file__).resolve().parent.parent
    DATA = ROOT / "data"
    FIG = ROOT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

# Target printed width. The LaTeX text block is 6.27 in wide, so a figure drawn
# much wider than this is downscaled by \includegraphics and every font in it
# shrinks with the drawing -- the usual reason a figure set looks amateur on the
# page. Everything here is laid out at TW and lands at ~0.9 scale.
TW = 6.9

# ---- palette -------------------------------------------------------------
# A cool, low-chroma set in the register of a printed journal figure: navy,
# wine and violet rather than the primary blue / orange / green of a screen
# palette. The three model arms stay separable for the common forms of colour
# blindness -- they differ in hue *and* in lightness (navy 30%, wine 42%,
# violet 55%), so they also survive a greyscale print -- and identity never
# rests on colour alone, since every series carries a marker or a direct label.
PLS     = "#24486b"   # categorical slot 1 -- navy
CVAE_T  = "#8f3a4f"   # slot 2 -- wine
CVAE_N  = "#6f6396"   # slot 3 -- muted violet
AD      = "#8f3a4f"   # diverging warm pole -- wine
CTRL    = "#24486b"   # diverging cool pole -- navy
UP, DOWN = AD, CTRL
MID     = "#8a8a92"   # neutral midpoint ink, faintly cool
ACCENT  = "#2f7d8c"   # reserved for "found by both" -- petrol blue
INK     = "#17171c"
INK2    = "#54545e"
INK3    = "#8a8a92"
RULE    = "#c9c9d1"
GRID    = "#e9e9ef"
SURFACE = "#ffffff"

# Diverging ramp for probability surfaces: cool pole -> neutral grey -> warm
# pole. Deliberately light at both ends -- this ramp is only ever a background
# under data marks, and a saturated pole swallows them.
DIVERGING = LinearSegmentedColormap.from_list(
    "ctrl_ad", ["#9aabc0", "#bcc6d5", "#dadfe7", "#eeeef1",
                "#e8d3d9", "#d5aeb9", "#bf8b9a"])

# Sequential ramps for the confusion matrices, built from the two arm colours so
# the matrices sit inside the same muted palette as everything else instead of
# borrowing matplotlib's saturated "Blues"/"Oranges".
SEQ_PLS = LinearSegmentedColormap.from_list(
    "seq_pls", ["#ffffff", "#e7ecf2", "#c3d1e0", "#93aac6", "#5c7fa6", PLS, "#173252"])
SEQ_CVAE = LinearSegmentedColormap.from_list(
    "seq_cvae", ["#ffffff", "#f5e9ec", "#e6c9d1", "#d0a1ae", "#b06c81", CVAE_T, "#63242f"])

MODEL_COLOR = {"pls": PLS, "cvae_true": CVAE_T, "cvae_neutral": CVAE_N,
               "PLS-DA": PLS, "CVAE_true": CVAE_T, "CVAE_neutral": CVAE_N}
MODEL_LABEL = {"pls": "PLS-DA",
               "cvae_true": "CVAE (label-conditioned)",
               "cvae_neutral": "CVAE (label-agnostic)",
               "PLS-DA": "PLS-DA",
               "CVAE_true": "CVAE (label-conditioned)",
               "CVAE_neutral": "CVAE (label-agnostic)"}
GROUP_COLOR = {"AD": AD, "Control": CTRL}
GROUP_MARKER = {"AD": "^", "Control": "o"}

# database sources in the g:Profiler output
SRC_COLOR = {"GO:MF": "#24486b", "GO:BP": "#2f7d8c", "GO:CC": "#6f6396",
             "CORUM": "#8f3a4f", "KEGG": "#4b7ca8", "REAC": "#57516e",
             "WP": "#3f6d80", "MIRNA": "#a3798f", "TF": "#54545e", "HP": "#7a4a5e"}

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral", "Latin Modern Roman", "DejaVu Serif", "Times New Roman"],
    "mathtext.fontset": "stix",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.titlelocation": "left",
    "axes.titlepad": 7,
    "axes.labelsize": 8,
    "axes.labelpad": 3.5,
    "axes.labelcolor": INK,
    "axes.edgecolor": RULE,
    "axes.linewidth": 0.6,
    "axes.grid": False,
    "grid.color": GRID,
    "grid.linewidth": 0.5,
    "axes.axisbelow": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.color": INK2, "ytick.color": INK2,
    "xtick.labelsize": 7, "ytick.labelsize": 7,
    "xtick.major.size": 2.6, "ytick.major.size": 2.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.pad": 2.5, "ytick.major.pad": 2.5,
    "legend.fontsize": 7,
    "legend.frameon": False,
    "legend.handlelength": 1.1,
    "legend.handletextpad": 0.5,
    "legend.columnspacing": 1.2,
    "legend.borderaxespad": 0.2,
    "lines.linewidth": 1.4,
    "lines.solid_capstyle": "round",
    "patch.linewidth": 0,
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "pdf.fonttype": 42,
})


# ---- primitives ----------------------------------------------------------
def save(fig, name):
    """Vector PDF for pdflatex, 400-dpi PNG for the pandoc/docx build."""
    for ext, kw in (("pdf", {}), ("png", {"dpi": 400})):
        fig.savefig(FIG / f"{name}.{ext}", bbox_inches="tight",
                    pad_inches=0.03, **kw)
    if INLINE:
        try:
            from IPython.display import display
            display(fig)
        except ImportError:      # running the notebook's cells outside a kernel
            pass
    plt.close(fig)
    print(f"  {FIG.name}/{name}.pdf + .png")


def panel(ax, letter, title=None, subtitle=None, dx=-0.13, dy=1.0):
    """Panel letter in sans-serif, hung outside the axes at a fixed offset so
    the letters line up down the figure regardless of tick-label width.

    Title and subtitle are both annotations rather than `set_title`, because a
    real title always sits at one fixed height and would land on top of the
    subtitle whenever both are present.
    """
    ax.annotate(letter, xy=(dx, dy), xycoords="axes fraction",
                xytext=(0, 2), textcoords="offset points",
                fontsize=9.5, fontweight="bold", family="DejaVu Sans",
                color=INK, ha="left", va="bottom", annotation_clip=False)
    if title:
        ax.annotate(title, xy=(0, 1.0), xycoords="axes fraction",
                    xytext=(0, 13 if subtitle else 3), textcoords="offset points",
                    fontsize=8.6, color=INK, ha="left", va="bottom",
                    annotation_clip=False)
    if subtitle:
        ax.annotate(subtitle, xy=(0, 1.0), xycoords="axes fraction",
                    xytext=(0, 3), textcoords="offset points",
                    fontsize=6.8, color=INK3, ha="left", va="bottom",
                    annotation_clip=False)


def ygrid(ax):
    ax.grid(axis="y", visible=True)
    ax.grid(axis="x", visible=False)


def xgrid(ax):
    ax.grid(axis="x", visible=True)
    ax.grid(axis="y", visible=False)


def despine(ax, left=True, bottom=True):
    ax.spines["left"].set_visible(left)
    ax.spines["bottom"].set_visible(bottom)


def load(name):
    return pd.read_csv(DATA / name)


def scal(df=None):
    """summary_scalars.csv -> dict, numeric where possible."""
    df = df if df is not None else load("summary_scalars.csv")
    out = {}
    for _, r in df.iterrows():
        try:
            out[r["key"]] = float(r["value"])
        except (TypeError, ValueError):
            out[r["key"]] = r["value"]
    return out


def stars(p):
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else "n.s."


def pmath(p, prefix="p"):
    """`$p = ...$` with a real times-ten-to-the exponent, not `1e-05`."""
    if p >= 1e-3:
        return f"${prefix} = {p:.3f}$"
    if not p > 1e-300:          # V7: with thousands of pairs a p-value can underflow to 0
        return f"${prefix} < 10^{{-300}}$"
    e = int(np.floor(np.log10(p)))
    return f"${prefix} = {p/10**e:.1f}\\times 10^{{{e}}}$"


def italic_ticks(ax, names, fontsize=6.0, axis="y"):
    """Italic gene symbols as real text, so hyphens stay hyphens and digits do
    not pick up math-italic spacing (`HLA-DQA2`, not `HLA - DQA2`)."""
    ticks = np.arange(len(names))
    if axis == "y":
        ax.set_yticks(ticks)
        ax.set_yticklabels(names, fontsize=fontsize, fontstyle="italic")
    else:
        ax.set_xticks(ticks)
        ax.set_xticklabels(names, fontsize=fontsize, fontstyle="italic")


def bracket(ax, x0, x1, y, text, tick=None, fontsize=7.2, color=INK):
    """Significance bracket that never overlaps the marks it spans."""
    tick = tick if tick is not None else abs(y) * 0.02 + 1e-9
    ax.plot([x0, x0, x1, x1], [y - tick, y, y, y - tick],
            color=color, lw=0.7, clip_on=False, solid_capstyle="butt")
    ax.text((x0 + x1) / 2, y + tick * 0.5, text, ha="center", va="bottom",
            fontsize=fontsize, color=color, clip_on=False)


def boxstrip(ax, values, pos, color, width=0.44, jitter=0.055, seed=0, size=4):
    """A box over its own points: the distribution and its summary together."""
    rs = np.random.RandomState(int(round(seed)))
    ax.scatter(rs.normal(pos, jitter, len(values)), values, s=size, color=color,
               alpha=0.28, linewidth=0, zorder=1, rasterized=True)
    bp = ax.boxplot([values], positions=[pos], widths=width, showfliers=False,
                    patch_artist=True, zorder=3, manage_ticks=False)
    bp["boxes"][0].set(facecolor=SURFACE, edgecolor=color, linewidth=1.2, alpha=0.95)
    for part in ("whiskers", "caps", "medians"):
        for a in bp[part]:
            a.set(color=color, linewidth=1.2)
    return bp


def swatches(labels_colors, marker="s", size=5):
    return [Line2D([], [], marker=marker, ls="", color=c, markersize=size, label=l)
            for l, c in labels_colors]


def repel_labels(ax, xs, ys, labels, fontsize=5.4, color=None, italic=True,
                 clearance=1.6, leader_color=None):
    """Place a label for every point without letting any two labels touch.

    A dependency-free replacement for adjustText: each label is tried at a ring
    of candidate offsets, in decreasing order of preference, and the first
    offset whose estimated box overlaps neither an already-placed label nor its
    own point is taken. Everything is measured in typographic points, which is
    exactly the unit `annotate` takes for `xytext`, so no round trip through
    data coordinates is needed and the result is stable at any figure size.
    """
    color = INK if color is None else color
    leader_color = RULE if leader_color is None else leader_color
    fig = ax.figure
    ppd = fig.dpi / 72.0                     # points -> display pixels
    pts = ax.transData.transform(np.column_stack([xs, ys]))

    # candidate offsets: near ring first, then progressively further out
    cands = []
    for radius in (7, 11, 16, 22, 30, 40, 52, 66, 82, 100):   # V7: wider search
        for ang in range(0, 360, 30):
            a = np.deg2rad(ang)
            cands.append((radius * np.cos(a), radius * np.sin(a)))

    placed = []                              # display-space boxes already taken
    leg = ax.get_legend()                    # a label under the key is unreadable
    if leg is not None:
        bb = leg.get_window_extent()
        placed.append((bb.x0, bb.y0, bb.x1, bb.y1))
    order = np.argsort(-np.asarray(ys))      # most significant genes get first pick

    def overlaps(box):
        x0, y0, x1, y1 = box
        for p0, q0, p1, q1 in placed:
            if x0 < p1 and p0 < x1 and y0 < q1 and q0 < y1:
                return True
        return False

    axbb = ax.get_window_extent()            # V7: labels must stay inside the axes
    inside = lambda b: (b[0] >= axbb.x0 and b[2] <= axbb.x1 and
                        b[1] >= axbb.y0 and b[3] <= axbb.y1)
    forced = 0
    for i in order:
        text = labels[i]
        w = len(text) * fontsize * 0.56 * ppd + clearance * ppd
        h = fontsize * 1.25 * ppd + clearance * ppd
        px, py = pts[i]
        boxes = []
        for dx, dy in cands:
            cx = px + dx * ppd + (w / 2 - clearance) * (1 if dx >= 0 else -1)
            cy = py + dy * ppd
            boxes.append(((cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2), dx, dy))
        ok = [t for t in boxes if inside(t[0]) and not overlaps(t[0])]
        if not ok:                           # never drop a gene: least-bad inside spot
            forced += 1
            ok = [t for t in boxes if inside(t[0])] or boxes
        for box, dx, dy in ok[:1]:
            if True:
                placed.append(box)
                ax.annotate(text, (xs[i], ys[i]), xytext=(dx, dy),
                            textcoords="offset points", fontsize=fontsize,
                            color=color, fontstyle="italic" if italic else "normal",
                            ha="left" if dx >= 0 else "right", va="center",
                            zorder=5, annotation_clip=False,
                            arrowprops=dict(arrowstyle="-", color=leader_color,
                                            lw=0.35, shrinkA=0.5, shrinkB=1.5))
    if forced:
        print(f"  repel_labels: {forced} label(s) had no collision-free spot")


def kde2d(xy, grid_x, grid_y, bw_scale=1.0):
    """Gaussian KDE evaluated on a grid, without a scipy import at call sites."""
    from scipy.stats import gaussian_kde
    k = gaussian_kde(xy.T)
    k.set_bandwidth(k.factor * bw_scale)
    XX, YY = np.meshgrid(grid_x, grid_y)
    return k(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)

In [ ]:
# =====================================================================
# Figure E1 -- external prediction on the held-out cohort
# =====================================================================
def fig_external_prediction():
    runs = load("loco_run_metrics.csv"); roc = load("loco_roc_curves.csv")
    mp = load("external_mean_predictions.csv")
    arms = [("pls", "PLS-DA", PLS, "-"), ("cvae_neutral", "CVAE (label-agnostic)", CVAE_N, "-"),
            ("cvae_true", "CVAE (label-conditioned)*", CVAE_T, (0, (2, 1.5))),
            ("age", "age only", MID, (0, (4, 2)))]
    fig = plt.figure(figsize=(TW, 6.4))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 0.9], width_ratios=[1.0, 1.0],
                          hspace=0.55, wspace=0.34)
    ax = fig.add_subplot(gs[0, 0]); base = np.linspace(0, 1, 101)
    for m, lab, col, ls in arms:
        v = roc[roc.model == m][[f"tpr_{i}" for i in range(101)]].values
        mu, sd = v.mean(0), v.std(0)
        if m != "cvae_true":
            ax.fill_between(base, mu - sd, mu + sd, color=col, alpha=0.12, lw=0)
        ax.plot(base, mu, color=col, lw=1.4 if m != "cvae_true" else 1.0, ls=ls,
                label=f"{lab}  {runs[f'auc_{m}'].mean():.3f}")
    ax.plot([0, 1], [0, 1], color=INK3, lw=0.6, ls=(0, (3.5, 2.5)))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.004); ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.legend(loc="lower right", title="mean ROC-AUC", title_fontsize=6.4, fontsize=6.2,
              alignment="left", frameon=True, framealpha=0.92, facecolor=SURFACE, edgecolor="none")
    panel(ax, "a", "External cohort ROC", "mean $\\pm$ SD over 20 training runs")

    ax = fig.add_subplot(gs[0, 1])
    for i, (m, lab, col, _) in enumerate(arms):
        boxstrip(ax, runs[f"auc_{m}"].values, i, col, width=0.5, seed=i, size=3)
    ax.axhline(0.5, color=INK3, lw=0.7, ls=(0, (3.5, 2.5)))
    ax.set_xticks(range(len(arms)))
    ax.set_xticklabels(["PLS-DA", "CVAE\nagnostic", "CVAE\nconditioned*", "age\nonly"], fontsize=6.6)
    ax.set_ylabel("ROC-AUC"); ax.set_ylim(0.3, 1.02); ygrid(ax)
    panel(ax, "b", "One point per training run", "the same 20 runs for every arm")

    ax = fig.add_subplot(gs[1, :])
    from scipy.stats import gaussian_kde
    grid = np.linspace(0, 1, 300)
    for m, lab, ls in (("p_pls", "PLS-DA", "-"), ("p_cvae_neutral", "CVAE (label-agnostic)", (0, (4, 2)))):
        for g in ("Control", "AD"):
            v = mp.loc[mp.group == g, m].values
            d = gaussian_kde(v)(grid)
            ax.plot(grid, d, color=GROUP_COLOR[g], lw=1.4, ls=ls)
            if ls == "-":
                ax.fill_between(grid, 0, d, color=GROUP_COLOR[g], alpha=0.10, lw=0)
    ax.axvline(0.5, color=INK3, lw=0.7, ls=(0, (3.5, 2.5)))
    ax.set_xlim(0, 1); ax.set_ylim(0, ax.get_ylim()[1] * 1.35)
    ax.set_xlabel("Predicted $P(\\mathrm{AD})$, mean over the 20 runs"); ax.set_ylabel("Density"); ygrid(ax)
    ax.legend(handles=[*swatches([("Control", CTRL), ("AD", AD)]),
                       Line2D([], [], color=INK2, ls="-", label="PLS-DA"),
                       Line2D([], [], color=INK2, ls=(0, (4, 2)), label="CVAE (label-agnostic)")],
              loc="upper center", ncol=4, fontsize=6.4, frameon=True, framealpha=0.92,
              facecolor=SURFACE, edgecolor="none")
    panel(ax, "c", "Predicted risk on the external donors", f"n = {len(mp)} donors", dx=-0.065)
    fig.text(0.5, 0.0, "* given each external array's true diagnosis as an input; reference only.",
             ha="center", fontsize=6.4, color=INK2, style="italic")
    save(fig, "figE1_external_prediction")


fig_external_prediction()


In [ ]:
# =====================================================================
# Figure E2 -- age validation
# =====================================================================
def tint(hex_color, alpha):
    """Flat tint over the white surface (same as the Figure 1 helper)."""
    r, g, b = (int(hex_color[i:i + 2], 16) / 255 for i in (1, 3, 5))
    return (1 - alpha + alpha * r, 1 - alpha + alpha * g, 1 - alpha + alpha * b)


def fig_age_validation():
    mp = load("external_mean_predictions.csv"); bands = load("external_age_bands.csv")
    A = dict(zip(load("external_age_validation.csv").key, load("external_age_validation.csv").value))
    fig = plt.figure(figsize=(TW, 6.6))
    gs = fig.add_gridspec(2, 2, hspace=0.62, wspace=0.34)
    for i, (m, lab) in enumerate((("pls", "PLS-DA"), ("cvae_neutral", "CVAE (label-agnostic)"))):
        ax = fig.add_subplot(gs[0, i])
        for g in ("Control", "AD"):
            s = mp[mp.group == g]
            ax.scatter(s.age + np.random.RandomState(i).uniform(-0.3, 0.3, len(s)), s[f"p_{m}"],
                       s=11, color=GROUP_COLOR[g], marker=GROUP_MARKER[g], alpha=0.75,
                       edgecolor=SURFACE, linewidth=0.4, label=g)
        ax.set_ylim(-0.04, 1.30); ax.set_xlabel("Age at death (years; 90+ coded 90)")
        ax.set_ylabel("Predicted $P(\\mathrm{AD})$" if i == 0 else ""); ygrid(ax)
        txt = (f"controls $\\rho = {float(A[f'rho_age_{m}_control']):+.2f}$ ({pmath(float(A[f'rho_age_{m}_control_p']))})\n"
               f"AD $\\rho = {float(A[f'rho_age_{m}_AD']):+.2f}$ ({pmath(float(A[f'rho_age_{m}_AD_p']))})")
        ax.text(0.02, 0.975, txt, transform=ax.transAxes, va="top", fontsize=6.3, color=INK2)
        if i == 1:
            ax.legend(loc="upper right", fontsize=6.4, frameon=True, framealpha=0.92,
                      facecolor=SURFACE, edgecolor="none", markerscale=1.2)
        panel(ax, "ab"[i], f"{lab}: does risk track age?", "Spearman $\\rho$ within each group")

    ax = fig.add_subplot(gs[1, 0])
    arms = [("pls", "PLS-DA", PLS, "o"), ("cvae_neutral", "CVAE (agnostic)", CVAE_N, "s"),
            ("age", "age only", MID, "^")]
    x = np.arange(len(bands))
    for j, (m, lab, col, mk) in enumerate(arms):
        ax.errorbar(x + (j - 1) * 0.18, bands[f"auc_{m}_mean"], yerr=bands[f"auc_{m}_sd"], fmt=mk,
                    color=col, ms=5, lw=0.9, capsize=2, label=lab)
    ax.axhline(0.5, color=INK3, lw=0.7, ls=(0, (3.5, 2.5)))
    ax.set_xticks(x)
    ax.set_xticklabels([f"{b.band}\n{b.n_AD} AD / {b.n_control} C" for b in bands.itertuples()], fontsize=6.4)
    ax.set_ylim(0.2, 1.12); ax.set_ylabel("ROC-AUC"); ygrid(ax)
    ax.legend(loc="lower left", ncol=3, fontsize=6.0, frameon=True, framealpha=0.92,
              facecolor=SURFACE, edgecolor="none", columnspacing=0.8, handletextpad=0.3)
    panel(ax, "c", "Within age bands", "age is nearly constant inside each band")

    ax = fig.add_subplot(gs[1, 1])
    labels, vals, cols = ["age alone"], [float(A["auc_age_only_raw"])], [MID]
    for m, lab, col in (("pls", "PLS-DA", PLS), ("cvae_neutral", "CVAE", CVAE_N)):
        labels += [lab, f"{lab}\n+ age"]
        vals += [float(A[f"auc_{m}_mean_prob"]), float(A[f"auc_{m}_plus_age_insample"])]
        cols += [col, col]
    xs = np.arange(len(vals))
    bars = ax.bar(xs, vals, 0.62, color=[tint(c, 0.25) if "+ age" in l else c for c, l in zip(cols, labels)],
                  edgecolor=cols, linewidth=1.0)
    for xi, v in zip(xs, vals):
        ax.text(xi, v + 0.015, f"{v:.3f}", ha="center", fontsize=6.0, color=INK2)
    ax.set_xticks(xs); ax.set_xticklabels(labels, fontsize=6.2)
    ax.set_ylim(0.4, 1.08); ax.set_ylabel("ROC-AUC"); ygrid(ax)
    ax.text(0.02, 0.975, "model adds to age (likelihood-ratio test)\n"
            f"PLS-DA {pmath(float(A['lr_pls_beyond_age_p']))}\n"
            f"CVAE {pmath(float(A['lr_cvae_neutral_beyond_age_p']))}",
            transform=ax.transAxes, va="top", fontsize=6.2, color=INK2)
    ax.set_ylim(0.4, 1.22)
    panel(ax, "d", "Information beyond age", "AD ~ age  vs  AD ~ age + model score (in-sample)")
    save(fig, "figE2_age_validation")


fig_age_validation()


In [ ]:
# =====================================================================
# Figure E3 -- do the genes found on four cohorts hold in the fifth?
# =====================================================================
def fig_external_genes():
    g = load("loco_selected_genes.csv"); rep = load("loco_gene_replication.csv")
    sig = load("loco_signature_scores.csv"); ga = load("gene_age_association.csv")
    A = dict(zip(load("external_age_validation.csv").key, load("external_age_validation.csv").value))
    fig = plt.figure(figsize=(TW, 6.8))
    gs = fig.add_gridspec(2, 2, hspace=0.62, wspace=0.36)
    meth = [("PLS-DA", PLS, "o"), ("CVAE", CVAE_N, "s")]

    ax = fig.add_subplot(gs[0, 0])
    gd = g[g.direction_train != "none"]
    for m, col, mk in meth:
        s = gd[gd.method == m]
        sign = np.where(s.direction_train == "up", 1, -1)
        rep_ = s.replicated.astype(str).str.lower().eq("true").values
        ax.scatter(s.logFC_heldout.values * sign, sign + (0.12 if m == "CVAE" else -0.12)
                   + np.random.RandomState(1).uniform(-0.07, 0.07, len(s)),
                   s=14, marker=mk, facecolor=np.where(rep_, col, SURFACE), edgecolor=col,
                   linewidth=0.7, label=m, zorder=3)
    ax.axvline(0, color=INK3, lw=0.7, ls=(0, (3.5, 2.5)))
    ax.set_yticks([-1, 1]); ax.set_yticklabels(["down in AD\n(training)", "up in AD\n(training)"], fontsize=6.4)
    ax.set_ylim(-1.7, 1.9)
    ax.set_xlabel("External change in the trained direction\n(standardised AD $-$ control)")
    xgrid(ax)
    ax.legend(loc="upper left", ncol=2, fontsize=6.2, frameon=True, framealpha=0.92,
              facecolor=SURFACE, edgecolor="none")
    panel(ax, "a", "Each selected gene on the external cohort",
          "right of the line = same direction; filled = FDR $<0.05$")

    ax = fig.add_subplot(gs[0, 1])
    x = np.arange(2)
    for j, (m, col, mk) in enumerate(meth):
        r = rep[rep.method == m].iloc[0]
        for k, (obs, lo, hi, mu) in enumerate((("concordance", "null_concordance_lo", "null_concordance_hi", "null_concordance_mean"),
                                               ("replication", "null_replication_lo", "null_replication_hi", "null_replication_mean"))):
            xx = k + (j - 0.5) * 0.30
            ax.bar(xx, r[hi] - r[lo], bottom=r[lo], width=0.24, color=GRID, edgecolor=RULE, lw=0.5)
            ax.scatter(xx, r[obs], marker=mk, s=34, color=col, zorder=3, edgecolor=SURFACE,
                       linewidth=0.6, label=m if k == 0 else None)
            ax.text(xx, r[obs] + 0.05, f"{r[obs]:.0%}", ha="center", fontsize=6.0, color=INK2)
    ax.set_xticks(x); ax.set_xticklabels(["same direction", "same direction\nand FDR $<0.05$"], fontsize=6.6)
    ax.set_ylim(0, 1.15); ax.set_ylabel("fraction of selected genes"); ygrid(ax)
    ax.legend(loc="upper right", fontsize=6.2, frameon=True, framealpha=0.92,
              facecolor=SURFACE, edgecolor="none")
    panel(ax, "b", "Replication vs random gene sets", "grey = 95% range of 2,000 random sets")

    ax = fig.add_subplot(gs[1, 0])
    xs = np.arange(len(meth))
    for j, (m, col, mk) in enumerate(meth):
        s = sig[sig.method == m].iloc[0]
        ax.bar(j, s.signature_auc, 0.55, color=col, edgecolor=SURFACE)
        ax.text(j, np.nan_to_num(s.signature_auc) + 0.015,
                f"{s.signature_auc:.3f}\n{int(np.nan_to_num(s.n_up))} up / {int(np.nan_to_num(s.n_down))} down",
                ha="center", va="bottom", fontsize=6.0, color=INK2)
    ax.axhline(0.5, color=INK3, lw=0.7, ls=(0, (3.5, 2.5)))
    ax.set_xticks(xs); ax.set_xticklabels([m for m, _, _ in meth])
    ax.set_ylim(0.3, 1.12); ax.set_ylabel("ROC-AUC"); ygrid(ax)
    panel(ax, "c", "Gene set as a plain score", "mean(up) $-$ mean(down) on external arrays; no model")

    ax = fig.add_subplot(gs[1, 1])
    labs = ["all genes"] + [m for m, _, _ in meth]
    vals = [float(A["frac_age_genes_background"])] + [float(A.get(f"frac_age_genes_{m}", np.nan)) for m, _, _ in meth]
    cols = [MID] + [c for _, c, _ in meth]
    ax.bar(range(3), vals, 0.55, color=cols, edgecolor=SURFACE)
    for i, (v, m) in enumerate(zip(vals, labs)):
        extra = "" if i == 0 else f"\nFisher {pmath(float(A.get(f'age_genes_{m}_fisher_p', np.nan)))}"
        ax.text(i, v + 0.01, f"{v:.1%}{extra}", ha="center", va="bottom", fontsize=6.0, color=INK2)
    ax.set_xticks(range(3)); ax.set_xticklabels(labs)
    ax.set_ylim(0, max(vals) * 1.6 + 0.02); ax.set_ylabel("age-associated (FDR $<0.05$)"); ygrid(ax)
    panel(ax, "d", "Are the selected genes ageing genes?",
          "age correlation in the training cohorts' controls")
    save(fig, "figE3_external_genes")


fig_external_genes()


---

# Output manifest


In [ ]:
# every artefact this notebook wrote, as a single checkable manifest
import os

rows = []
for root, _, files in os.walk(OUT):
    for f in sorted(files):
        if f.startswith("__") or f.endswith(".ipynb"):
            continue
        p = os.path.join(root, f)
        rows.append({"file": os.path.relpath(p, OUT),
                     "bytes": os.path.getsize(p)})

manifest = pd.DataFrame(rows).sort_values("file").reset_index(drop=True)
manifest.to_csv(f"{OUT}/output_manifest.csv", index=False)
print(manifest.to_string(index=False))

empty = manifest[manifest.bytes == 0]
print(f"\n{len(manifest)} files, {manifest.bytes.sum()/1e6:.1f} MB")
if len(empty):
    print("EMPTY FILES -- something failed to render:")
    print(empty.to_string(index=False))
else:
    print("no empty files")